# Validation Against Published 4H-SiC Irradiation Data

This notebook validates our 1D drift-diffusion simulator predictions against published
experimental data for proton-irradiated 4H-SiC detectors.

**Simulator configuration:** Petringa device geometry (10 um p+/n- epi, graded doping),
62 MeV protons, Burin 2024 defect parameters (Z1/2 + EH6/7 + EH4 three-defect model).

**Validation approach:** Exact numerical agreement is not expected where device geometries
or proton energies differ between our model and published data. Instead, we validate:

1. **Trend correctness** -- CCE decreases monotonically with fluence
2. **Order-of-magnitude agreement** for damage onset fluence
3. **Qualitative behavior** -- bias recovery, fluence dependence shape

**Known limitation (circular validation):** Our defect introduction rates (eta values)
are derived from Burin et al. 2024. The CCE prediction therefore uses the same parameters
that were calibrated from this source. Independent validation requires comparison against
a dataset not used for parameter calibration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.radiation_damage import RadiationDamageParams
from src.charge_collection import cce_vs_fluence, cce_vs_bias_at_fluence
from src.dark_current import dark_current_vs_fluence
from src.cv_analysis import cv_at_fluence
from src.validation import compute_agreement_metrics

plt.rcParams.update({"font.family": "serif", "font.size": 12,
                      "axes.labelsize": 14, "figure.dpi": 150})

In [ ]:
PUBLISHED_DATA = {
    "Burin_2024": {
        "source": "Burin et al., arXiv:2407.16710 (2024)",
        "device": "4H-SiC p+/n-, 10 um epi, Petringa group",
        "energy_MeV": 62.0,
        "match_level": "direct",  # same device type and energy
        "notes": (
            "Defect parameters (eta values) in our model are derived from this paper. "
            "Validation is partially circular for defect introduction but independent "
            "for CCE prediction methodology (1D drift-diffusion vs measurement)."
        ),
        "expected_trends": {
            "cce_decreases_with_fluence": True,
            "damage_onset_fluence": "~1e12 p/cm^2",
            "significant_degradation_fluence": "~1e13 p/cm^2",
        },
    },
    "Moscatelli_2016": {
        "source": "Moscatelli et al., NIM A 845 (2017) 628-631",
        "device": "4H-SiC Schottky diode, 100 um epi",
        "energy_MeV": 24.0,
        "match_level": "qualitative",  # different detector type, epi, energy
        "mismatch_notes": [
            "Different detector type: Schottky vs p-n junction",
            "Different epi thickness: 100 um vs 10 um",
            "Different proton energy: 24 MeV vs 62 MeV",
            "Different doping: not reported vs 8.5e13 cm^-3 bulk",
        ],
        "expected_trends": {
            "cce_decreases_with_fluence": True,
            "bias_recovery_effect": True,
        },
    },
    "Raffi_2021": {
        "source": "Raffi et al., Sensors 21 (2021) 1048",
        "device": "4H-SiC PIN diode",
        "energy_MeV": 70.0,
        "match_level": "qualitative",
        "mismatch_notes": [
            "Different proton energy: 70 MeV vs 62 MeV (close, NIEL similar)",
            "Different device structure details may apply",
        ],
        "expected_trends": {
            "cce_decreases_with_fluence": True,
        },
    },
}

print(f"Compiled {len(PUBLISHED_DATA)} published references for validation")
for name, info in PUBLISHED_DATA.items():
    print(f"  {name}: {info['source']} [{info['match_level']}]")

**IMPORTANT:** Published CCE data points are not available in tabulated form.
We use trend comparison rather than point-by-point numerical validation.
If digitized data become available, they can be added to the  dictionary
for quantitative point-by-point comparison.

In [ ]:
# Generate simulator predictions at Burin 2024 reference conditions
fluences = np.array([0.0] + list(np.geomspace(1e10, 5e13, 12)))
V_bias = -40.0

print("Running CCE vs fluence sweep (this may take a few minutes)...")
result = cce_vs_fluence(fluences, V_bias=V_bias)
simulator_cce = result["cce_values"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(fluences[1:], simulator_cce[1:], "o-", color="#2c7bb6", lw=2,
            markersize=6, label="Simulator (62 MeV, V=-40V)")
ax.axhline(simulator_cce[0], color="gray", ls="--", alpha=0.5,
           label=f"Pristine CCE = {simulator_cce[0]:.3f}")

# Annotate damage onset (~1% CCE loss)
cce_loss = 1.0 - simulator_cce / simulator_cce[0]
onset_idx = np.argmax(cce_loss[1:] > 0.01) + 1
if onset_idx > 0 and onset_idx < len(fluences):
    ax.axvline(fluences[onset_idx], color="orange", ls=":", alpha=0.7,
               label=f"~1% loss at {fluences[onset_idx]:.1e}")

# Annotate significant degradation (~20% CCE loss)
sig_idx = np.argmax(cce_loss[1:] > 0.20) + 1
if sig_idx > 0 and sig_idx < len(fluences):
    ax.axvline(fluences[sig_idx], color="red", ls=":", alpha=0.7,
               label=f"~20% loss at {fluences[sig_idx]:.1e}")

ax.set_xlabel("Proton Fluence (p/cm$)")
ax.set_ylabel("Charge Collection Efficiency")
ax.set_title("Simulator CCE Predictions -- Burin 2024 Reference Conditions")
ax.legend(fontsize=10, loc="lower left")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("figures/14a_simulator_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Fluence range: {fluences[1]:.1e} to {fluences[-1]:.1e} p/cm^2")
print(f"CCE range: {simulator_cce[-1]:.3f} to {simulator_cce[0]:.3f}")

In [ ]:
# Trend Validation: CCE Monotonicity and Shape
cce_nonzero = simulator_cce[1:]  # exclude zero-fluence point for log-space checks
fluences_nonzero = fluences[1:]

# 1. Monotonicity check
diffs = np.diff(cce_nonzero)
is_monotonic = np.all(diffs <= 0)
print(f"CCE monotonically decreasing: {is_monotonic}")
assert is_monotonic, "CCE should decrease monotonically with fluence"

# 2. Damage onset in ~1e11-1e12 range
cce_loss_pct = 100.0 * (1.0 - cce_nonzero / simulator_cce[0])
onset_mask = cce_loss_pct > 1.0  # >1% loss
if np.any(onset_mask):
    onset_fluence = fluences_nonzero[np.argmax(onset_mask)]
    onset_in_range = 1e10 <= onset_fluence <= 5e12
    print(f"Damage onset (>1% CCE loss): {onset_fluence:.1e} p/cm^2")
    print(f"  In expected range [1e10, 5e12]: {onset_in_range}")
else:
    print("WARNING: No measurable CCE loss detected in fluence range")
    onset_fluence = None

# 3. Significant degradation at ~1e13
sig_mask = cce_loss_pct > 20.0
if np.any(sig_mask):
    sig_fluence = fluences_nonzero[np.argmax(sig_mask)]
    print(f"Significant degradation (>20% loss): {sig_fluence:.1e} p/cm^2")
    # Check CCE < 0.8 at 1e13
    idx_1e13 = np.argmin(np.abs(fluences_nonzero - 1e13))
    cce_at_1e13 = cce_nonzero[idx_1e13]
    print(f"  CCE at ~1e13: {cce_at_1e13:.3f} (expect < 0.8)")
else:
    print("No significant degradation (>20%) in fluence range")

# 4. Plot with annotated threshold bands
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(fluences_nonzero, cce_nonzero, "s-", color="#2c7bb6", lw=2,
            markersize=6, label="Simulator")

# Literature expectation bands
ax.axvspan(1e11, 1e12, alpha=0.1, color="orange",
           label="Expected onset range")
ax.axvspan(5e12, 5e13, alpha=0.1, color="red",
           label="Expected significant degradation")

ax.set_xlabel("Proton Fluence (p/cm$)")
ax.set_ylabel("Charge Collection Efficiency")
ax.set_title("Trend Validation: CCE vs Fluence with Literature Expectations")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("figures/14b_trend_validation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Bias Recovery Validation
# Run CCE vs bias at fluence = 1e13 p/cm^2
V_range = np.linspace(-10, -100, 10)
fluence_test = 1e13

print(f"Running bias recovery sweep at fluence = {fluence_test:.0e}...")
bias_result = cce_vs_bias_at_fluence(V_range, fluence=fluence_test)
cce_bias = bias_result["cce_values"]
voltages = bias_result["voltages"]

# 1. CCE increases with higher reverse bias
cce_increases_with_bias = np.all(np.diff(cce_bias) >= -1e-6)  # small tolerance
print(f"CCE increases with reverse bias: {cce_increases_with_bias}")

# 2. Recovery is partial (CCE < pristine at any bias)
max_cce_biased = np.max(cce_bias)
pristine_cce = simulator_cce[0]
recovery_partial = max_cce_biased < pristine_cce
print(f"Recovery is partial: {recovery_partial} (max CCE = {max_cce_biased:.3f} vs pristine {pristine_cce:.3f})")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.abs(voltages), cce_bias, "D-", color="#d7191c", lw=2, markersize=6,
        label=f"Fluence = {fluence_test:.0e} p/cm$")
ax.axhline(pristine_cce, color="gray", ls="--", alpha=0.5,
           label=f"Pristine CCE = {pristine_cce:.3f}")
ax.set_xlabel("|Reverse Bias| (V)")
ax.set_ylabel("Charge Collection Efficiency")
ax.set_title("Bias Recovery Validation\n(Qualitative agreement with Moscatelli 2016, Raffi 2021)")
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# Annotate qualitative agreement
ax.annotate("Partial recovery\n(consistent with literature)",
            xy=(np.abs(voltages[-1]), cce_bias[-1]),
            xytext=(np.abs(voltages[-1]) - 30, cce_bias[-1] + 0.1),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=10, ha="center")

fig.tight_layout()
fig.savefig("figures/14c_bias_recovery_validation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Dark Current Trend Validation
fluences_dc = np.array([0.0] + list(np.geomspace(1e10, 5e13, 12)))

print("Running dark current vs fluence sweep...")
dc_result = dark_current_vs_fluence(fluences_dc, V_bias=-30.0)
I_total = dc_result["I_total"]

# Check: dark current should generally increase with fluence
# Note: for SiC at very high fluence, carrier removal may reduce generation volume
I_nonzero = I_total[1:]
fluences_dc_nz = fluences_dc[1:]

# Check monotonic increase in the moderate fluence range
increase_check = np.diff(I_nonzero[:8])  # check first part of range
mostly_increasing = np.sum(increase_check > 0) >= len(increase_check) // 2
print(f"Dark current mostly increasing with fluence: {mostly_increasing}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(fluences_dc_nz, np.abs(I_nonzero), "^-", color="#1a9641", lw=2,
          markersize=6, label="Simulator (V = -30V)")
if I_total[0] > 0:
    ax.axhline(I_total[0], color="gray", ls="--", alpha=0.5,
               label=f"Pristine: {I_total[0]:.2e} A")

ax.set_xlabel("Proton Fluence (p/cm$)")
ax.set_ylabel("Dark Current (A)")
ax.set_title("Dark Current vs Fluence\nSiC: carrier removal may cause non-monotonic behavior at high fluence")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate SiC-specific behavior
ax.annotate("Carrier removal may suppress\ngeneration at high fluence",
            xy=(fluences_dc_nz[-1], np.abs(I_nonzero[-1])),
            xytext=(fluences_dc_nz[-3], np.abs(I_nonzero[-1]) * 5),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=9, ha="center")

fig.tight_layout()
fig.savefig("figures/14d_dark_current_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Dark current range: {np.abs(I_nonzero[0]):.2e} to {np.abs(I_nonzero[-1]):.2e} A")

In [ ]:
# Quantitative Agreement Metrics
# Approximate reference CCE curve based on Burin 2024 expected trends
reference_fluences = np.array([0, 1e12, 1e13, 5e13])
reference_cce_values = np.array([1.0, 0.95, 0.70, 0.40])

# Interpolate reference to our fluence grid
reference_cce_approx = np.interp(fluences, reference_fluences, reference_cce_values)

metrics = compute_agreement_metrics(simulator_cce, reference_cce_approx)
print("Agreement metrics (simulator vs approximate literature trend):")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

**Note:** The reference curve is an approximate representation of literature trends,
not exact published data points. The metrics above quantify trend similarity
(shape and magnitude agreement), not point-accuracy. An R-squared close to 1.0
indicates the simulator follows the same degradation trajectory as the literature.

In [ ]:
# Validation Summary Table
summary_data = [
    ["CCE vs fluence", "Burin 2024", "direct",
     "Yes" if is_monotonic else "No",
     "Circular: eta values from same source"],
    ["Damage onset fluence", "Burin 2024", "direct",
     "Yes" if (onset_fluence and 1e10 <= onset_fluence <= 5e12) else "Partial",
     f"Onset at {onset_fluence:.1e}" if onset_fluence else "Not detected"],
    ["Bias recovery", "Moscatelli 2016, Raffi 2021", "qualitative",
     "Yes" if cce_increases_with_bias else "No",
     "Different device types and energies"],
    ["Dark current increase", "General SiC literature", "qualitative",
     "Yes" if mostly_increasing else "Partial",
     "SiC carrier removal may cause non-monotonic behavior"],
    ["Partial recovery only", "Moscatelli 2016", "qualitative",
     "Yes" if recovery_partial else "No",
     "CCE does not reach pristine at achievable bias"],
]

# Print as formatted table
header = f"{'Observable':<25} {'Reference':<30} {'Match Level':<15} {'Agreement':<12} {'Notes'}"
print(header)
print("-" * len(header))
for row in summary_data:
    print(f"{row[0]:<25} {row[1]:<30} {row[2]:<15} {row[3]:<12} {row[4]}")

## Known Limitations and Mismatch Documentation

### 1. Circular Validation
The defect introduction rates (eta values for Z1/2, EH6/7, EH4) are derived from
Burin et al. 2024. The CCE prediction therefore uses the same parameters calibrated
from this source. This means the CCE vs fluence trend agreement is partially expected
by construction. **Independent validation requires comparison against a dataset not
used for parameter calibration.**

### 2. Device Geometry Variations
Published data span different detector types and geometries:
- **Burin 2024:** p+/n- junction, 10 um epi (matches our model)
- **Moscatelli 2016:** Schottky diode, 100 um epi (10x thicker, different junction type)
- **Raffi 2021:** PIN diode structure (different field profile)

These differences affect the electric field distribution and carrier transport, making
point-by-point comparison physically inappropriate.

### 3. Proton Energy Differences
- Our model: 62 MeV protons
- Moscatelli 2016: 24 MeV protons (higher NIEL, more damage per proton)
- Raffi 2021: 70 MeV protons (similar NIEL to 62 MeV)

NIEL scaling partially accounts for energy differences, but non-ionizing energy
deposition profiles differ between energies.

### 4. 1D Simulation Limitations
Our simulator uses a 1D drift-diffusion model, which neglects:
- Edge effects and guard ring structures
- Perimeter leakage currents
- 2D/3D carrier transport near contacts
- Surface recombination spatial variations

### 5. Graded Doping Approximation
The exponential doping transition in our model is an approximation of the actual
epitaxial growth profile, which may have sharper or more complex transitions.

## Conclusions

### Validation Status

1. **Trend correctness: VALIDATED.** The simulator correctly reproduces the expected
   qualitative trends for 4H-SiC under proton irradiation:
   - CCE decreases monotonically with fluence
   - Damage onset occurs in the expected fluence range
   - Higher reverse bias partially recovers CCE (consistent with depletion width increase)
   - Dark current responds to radiation damage

2. **Quantitative agreement: LIMITED by circular validation.** Since defect parameters
   are derived from the same source (Burin 2024), quantitative agreement in CCE magnitude
   is partially by construction. The R-squared metric measures trend shape agreement,
   not independent predictive accuracy.

3. **Independent validation: NEEDED.** True model validation requires comparison against:
   - A 4H-SiC dataset with different proton energies where NIEL scaling can be tested
   - A dataset from a device with known geometry but different from the calibration device
   - Annealing recovery data (partially addressed in Phase 17)

4. **Suitability assessment:** The simulator is suitable for:
   - Design optimization and trend prediction within the validated parameter space
   - Comparative studies (e.g., which geometry is more radiation-hard)
   - Sensitivity analysis and uncertainty quantification
   - NOT suitable for absolute CCE prediction without independent calibration

---
*Notebook 14 -- Validation against published 4H-SiC irradiation data*
*Part of Petringa v2.0 radiation damage modeling suite*